# Verificación post-auditoría de la capa de datos (5-jun)

Confirma en vivo los checks de `docs/auditoria_capa_datos_2026-06-05.md`. Una corrida,
serverless, sin re-escanear los 14 GB (solo agregados sobre la Gold ~1.33 GB y metadata).
NO exporta ni scorea.

**Nota MLflow (serverless / Free Edition):** la celda final llama
`set_tracking_uri("databricks")` y `set_registry_uri("databricks-uc")` ANTES de
`set_experiment`. Es obligatorio en serverless: sin esas dos líneas, `set_experiment` falla
con `CONFIG_NOT_AVAILABLE: spark.mlflow.modelRegistryUri`. NO las borres.

In [ ]:
from pyspark.sql import functions as F
GOLD   = "/Volumes/workspace/default/e_commerce/gold/features_session"
SILVER = "/Volumes/workspace/default/e_commerce/silver/clickstream_clean"
BRONZE = "/Volumes/workspace/default/e_commerce/bronze/clickstream"
gold = spark.read.format("delta").load(GOLD)
# Check 6 - esquema (esperado: 22 columnas)
print("n_cols:", len(gold.columns)); print(sorted(gold.columns))

In [ ]:
# Check 2 - grano (esperado: filas == sesiones distintas ~22.99M, 0 dup)
n = gold.count(); d = gold.select("user_session").distinct().count()
print("filas:", n, "| sesiones distintas:", d, "| grano OK:", n == d)

In [ ]:
# Check 1+3 - tasa de etiqueta (esperado full ~0.0610 / limpia ~0.0589)
gold.agg(F.avg("target_purchase").alias("tasa_full")).show()
gold.filter(F.col("label_window_corrupt") == 0).agg(F.avg("target_purchase").alias("tasa_limpia")).show()

In [ ]:
# Check 1 - flag sin_navegacion_previa (esperado ~34.144 filas, ~55% positivas)
gold.groupBy("sin_navegacion_previa").agg(F.count("*").alias("n"), F.avg("target_purchase").alias("tasa_pos")).show()

In [ ]:
# Check 3 - cuarentena + regimen de noviembre (15-nov~0, 16~0.057, 17~0.155; test 24-30)
gold.filter(F.col("session_date") >= "2019-11-01").groupBy("session_date") \
    .agg(F.count("*").alias("n"), F.avg("target_purchase").alias("tasa"), F.first("label_window_corrupt").alias("cuarentena")) \
    .orderBy("session_date").show(40, truncate=False)

In [ ]:
# Check 4 - categories_explored_cid poblada y distinta del macro
gold.agg(F.avg("categories_explored_cid").alias("avg_cid"), F.avg("categories_explored").alias("avg_macro"),
         F.sum(F.when(F.col("categories_explored_cid").isNull(), 1).otherwise(0)).alias("nulos_cid")).show()

In [ ]:
# Check 7 - particionamiento fisico (metadata, gratis)
for nombre, path in [("BRONZE", BRONZE), ("SILVER", SILVER), ("GOLD", GOLD)]:
    print(f"=== {nombre} ===")
    spark.sql(f"DESCRIBE DETAIL delta.`{path}`").select("partitionColumns", "numFiles", "sizeInBytes").show(truncate=False)

In [ ]:
# Idempotencia - sin Delta temporales en gold/ (esperado: [])
print([f.name for f in dbutils.fs.ls("/Volumes/workspace/default/e_commerce/gold/") if f.name.startswith("_tmp")])

In [ ]:
# MLflow smoke test - serverless: set_tracking_uri + set_registry_uri ANTES de set_experiment
import mlflow
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/pi_g8_modelado")
with mlflow.start_run(run_name="smoke_test"):
    mlflow.log_param("smoke", True); mlflow.log_metric("dummy", 1.0)
print(f"MLflow OK - tracking operativo (experiment: /Users/{user}/pi_g8_modelado)")